# Drowsy Guard – huấn luyện trên Colab
Runtime → Change runtime type → **GPU (T4)**.
1. Nén thư mục dự án thành `drowsy-guard.zip` rồi upload ở cell 2, hoặc `git clone` repo của bạn.
2. Tải MRL Eye Dataset (Kaggle) vào Drive hoặc dùng API Kaggle.

In [ ]:
!pip -q install mediapipe==0.10.14 onnx onnxruntime
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Cách A: upload zip
from google.colab import files
up = files.upload()  # chọn drowsy-guard.zip
!unzip -q -o drowsy-guard.zip -d /content
%cd /content/drowsy-guard
# Cách B: !git clone <URL_REPO> && %cd <TÊN_REPO>

In [ ]:
# Dữ liệu: mount Drive (đặt mrl-eye-dataset.zip vào MyDrive/datasets/)
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p data/raw/mrl && unzip -q -o /content/drive/MyDrive/datasets/mrl-eye-dataset.zip -d data/raw/mrl
# Tuỳ chọn: Kaggle API
# !pip -q install kaggle ; upload kaggle.json ; !kaggle datasets download -d imadeddinedjerarda/mrl-eye-dataset

In [ ]:
# Chia split theo subject (có thể thêm --own data/processed/own/eye_own.csv --holdout P07,P08)
!python training/prepare_data.py mrl --dir data/raw/mrl --out data/processed/eye_split.csv --cap 300

In [ ]:
!python training/train.py --csv data/processed/eye_split.csv --out models/eye.onnx --epochs 15 --pos_name closed

In [ ]:
# Mô hình mạnh hơn để so sánh (bảng đánh giá)
!python training/train.py --csv data/processed/eye_split.csv --out models/eye_mbv3.onnx --arch mobilenet --epochs 10 --pos_name closed

In [ ]:
# (Tuỳ chọn) Bộ yawn
# !python training/prepare_data.py yawn_folder --dir data/raw/yawn --out data/processed/yawn_split.csv
# !python training/train.py --csv data/processed/yawn_split.csv --out models/yawn.onnx --pos_name yawn

In [ ]:
from IPython.display import Image, display
display(Image('models/eye_cm.png'))
!zip -r models_out.zip models
from google.colab import files; files.download('models_out.zip')